# Lab 6: Neural network and multi-class classification
In this lab, you will train a neural network model with existing training data. You will learn how to train a multi-class classification neural network model and explore how different parameter settings in the neural network affect model performance. You will also learn how to do a multi-class accuracy assessment. 

The data you will be playing with this week is from [Dynamic World](https://www.dynamicworld.app/). It is a deep learning generated landcover classification dataset based on Sentinel-2 at 10m spatial resolution. Usually, the algorithm generated labels are also referred to as ‘weak labels’. The label can be downloaded from [Google Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1#description). The original Sentinel-2 image needs to be downloaded separately. To download the corresponding S2 images, search for the same image ID but from the sentinel-2 collection. Note that Dynamic world (DW) landcover classification label is generated based on [S2 TOA reflectance](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED#description) product, you can choose to train your model with the surface reflectance product, but its effect on model performance is not verified. Dynamic world is also near real-time product supported by Google Earth Engine, which means that the dynamic world labels of a scene will be made available almost as soon as the S2 TOA reflectance is available in GEE.  

The reason why I want to introduce DW in this lab is because it is a very important source of labeled data in remote sensing and the availability of labeled dataset is crucial in machine learning in general but is proven to be especially important for deep learning applications (not this lab). Even though dynamic world labels are algorithm generated and its accuracy is not as high as manually labeled datasets, but Dynamic world have ABUNDANT labeled data for model training. Many study shows that model trained with a large amount of low-quality label out performs models trained with limited amount of high-quality data. 

In actual applications, it does not hold much significance to use dynamic world labels to train a S2 model since those labels are generated through a deep learning model based on S2. It would be more useful to use DW label to train models based on other sensors. But since the focus of this lab is on training a NN model, we will use DW label as a demonstration.


## Step 1: Download DW and S2 images
The first part of the code that downloads DW and corresponding S2 image is exactly the same as lab 4, the trick here after acquiring the imgID of S2,we are searching for the img with samename but in a different collection. This is based on the description in the DW GEE page description: "Images in the Dynamic World collection have names matching the individual Sentinel-2 L1C asset names from which they were derived"

It is time consuming to select the suitable area for model training and validation. I have already downloaded suitable data and provided along with this lab. But the script is provided here for future reference.


In [2]:
import ee
# ee.Authenticate()

In [ ]:
# Initialize Earth Engine and import necessary libraries
import ee
ee.Initialize()
import geemap
import os

# Define your Area of Interest (AOI) shapefile
AOI_shp = r'./AOI_Shapefiles/Lab4_AOI_NorthAL.shp'  # Relative file path
AOI_geemap = geemap.shp_to_ee(AOI_shp)  # Convert shapefile to Earth Engine format

# Create an ImageCollection from the Dynamic World dataset
collection = (
    # ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterBounds(AOI_geemap)
    .filterDate('2019-10-18', '2019-12-18')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 1))
)

# Retrieve the list of image IDs from the collection
img_ids = collection.aggregate_array('system:id').getInfo()
print("The available S2 images within the AOI and defined time range are:\n", img_ids)

# Define the bands you want to download from the Dynamic World dataset
# dw_bands = ['water', 'trees', 'built']
S2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

# Prepare the download folder
download_folder = "./download_AL"
if not os.path.exists(download_folder):
    os.makedirs(download_folder)

# Loop through the images in the collection
for img_id in img_ids:
    # Download the Dynamic World image
    s2_image = ee.Image(img_id).select(S2_bands)
    s2_image_name = f"{img_id.split('/')[-1]}_s2.tif"
    s2_img_save_path = os.path.join(download_folder, s2_image_name)
    geemap.download_ee_image(
        s2_image,
        s2_img_save_path,
        region=AOI_geemap.geometry(),
        crs="EPSG:4326",
        scale=10,
    )
####
    # the previous steps are exactly the same as lab 4, the trick here is that we are searching for the img with samename but in a different collection
    # Extract the image identifier suffix
####
    image_suffix = img_id.split('/')[-1]  # e.g., "20190525T161901_20190525T163308_T16SFC"

    dw_image_id = f"GOOGLE/DYNAMICWORLD/V1/{image_suffix}"

    print(f"Corresponding Dynamic world image ID: {dw_image_id}")

    # Load the Sentinel-2 image
    try:
        dw_image = ee.Image(dw_image_id)

        # Define the Sentinel-2 bands you want to download (10m resolution bands)
        dw_bands = ['water', 'trees', 'built']  # Blue, Green, Red, NIR
        dw_image_to_download = dw_image.select(dw_bands)

        # Set the image name and save path
        dw_img_name = f"{image_suffix}_dw.tif"
        dw_img_save_path = os.path.join(download_folder, dw_img_name)

        # Download the Sentinel-2 image
        geemap.download_ee_image(
            dw_image_to_download,
            s2_img_save_path,
            region=AOI_geemap.geometry(),
            crs="EPSG:4326",
            scale=10,
        )
    except Exception as e:
        print(f"Failed to download Sentinel-2 image {dw_image_id}: {e}")


### Assignment 1: Explanation - Preprocessing DW Label Data

**Describe possible steps you would take to preprocess the DW label data to the same format I provided (4 classes in the image):**

The Dynamic World (DW) dataset provides probability bands for different land cover classes. Each band (e.g., 'water', 'trees', 'built') contains pixel values representing the estimated probability of complete coverage by that class (ranging from 0 to 1).

To convert this multi-band probability format into a single-band categorical format with 4 classes (0=other, 1=water, 2=trees, 3=built-up), the following steps would be taken:

1. **Load all DW probability bands**: Read all relevant probability bands from the DW image (water, trees, built, and potentially other classes like grass, flooded vegetation, crops, bare ground, snow/ice).

2. **Determine the dominant class for each pixel**: For each pixel, find the class with the highest probability value. This is typically done using `np.argmax()` across all probability bands.

3. **Map to categorical values**: Assign the dominant class to a categorical value:
   - Class with highest probability → assign corresponding class ID (0, 1, 2, or 3)
   - If the highest probability class is not one of the target classes (water, trees, built), assign it to class 0 (other)

4. **Optional thresholding**: Apply a probability threshold (e.g., 0.5) to ensure confidence - if the highest probability is below the threshold, assign to "other" class to avoid low-confidence classifications.

5. **Create single-band output**: Generate a single-band raster where each pixel contains only the categorical class value (0, 1, 2, or 3).

6. **Handle edge cases**: Ensure all pixels are assigned a valid class value, and handle any no-data or invalid probability values appropriately.

This approach converts the continuous probability values into discrete categorical labels suitable for supervised classification training.


## Step 2: Data Preprocessing
Each band in DW represent a certain land cover class and the values in that band indicates the estimated probability of complete coverage by that class. For example, if the band is water band, then the pixel values represent estimated probability of complete coverage by water.
However, this also means that we need to process the DW labels so that it only have one band and each pixel represents a certain landcover class. I have already processed the labels. Now the label only has one band and there are only 4 kind of values in the labels: 0, 1, 2, 3. Each pixel with value 0 represents other, 1 represents water, 2 represents trees, and 3 represents built-up areas.  

### Assignment 1
preprosess the training data I provided so that they are suitable for training a neural network. The chip size should be either 256, 128 or 64 depends on the the computation power you have. 

***push your code to process the data to github.***  

***Describe possible steps you would take to preprocess the DW label data to the same format I provided (4 classes in the image).***




## Step 3: NN model training
Scikit-learn handles multi-class classification model training pretty well, as long as you have pre-processed the labels correctly(eg. chipped, each pixel has a unique label), you do not need to specify the number of classes in advance during model training.

One feature that is special to neural network training is that you need to scale your data before training. This is because the weights in the neural network are updated based on the error between the predicted and actual values, and if the data is not scaled, the weights may update too slowly and the model may not converge to the optimal solution. Always scale your features when using neural networks. Methods like [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) or [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.minmax_scale.html) are commonly used. 

'StandardScaler()' is a class from sklearn.preprocessing that standardizes features by removing the mean and scaling to unit variance. When you create an instance of StandardScaler, you're preparing a scaler object that can compute the mean and standard deviation of your dataset. 'fit_transform(X)' is a method that first calculates the mean and standard deviation for each feature in your dataset X (this is the "fit" part). It then transforms the data by subtracting the mean and dividing by the standard deviation for each feature (this is the "transform" part). The result is a new array X_scaled, where each feature has a mean of 0 and a standard deviation of 1.

Shallow neural networks are very sensitive to the scale of the input data, so when you use trained model to predict new data, make sure to scale the input data using the same scaler object that was used to train the model. To do that, you will need to store the scaler object along with the model and use it to transform new data before making predictions.




When training a neural network with scikit-learn, you are actually calling the MLPClassifier class from scikit-learn. The MLPClassifier class has many parameters that you can set to control the model's architecture, learning rate, and other hyperparameters. Common parameters you can custermize include:


***hidden_layer_sizes:*** 

Specifies the number of neurons in the hidden layers.
Example: hidden_layer_sizes=(50, 30, 20) creates three hidden layers with 50, 30, and 20 neurons respectively. 

***activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'***  

Activation function for the hidden layers.
'identity': No-op activation, useful for implementing linear bottleneck.
'logistic': Logistic sigmoid function.
'tanh': Hyperbolic tan function.
'relu': Rectified linear unit function.  

***solver: {'lbfgs', 'sgd', 'adam'}, default='adam'***

This selects the optimization algorithm (what kind of gradient descent) to use for training the neural network.
'lbfgs': Quasi-Newton optimizer, good for smaller datasets.
'sgd': Stochastic gradient descent.
'adam': Stochastic gradient-based optimizer.  

***alpha: float, default=0.0001***  

L2 penalty (regularization term) parameter to prevent overfitting.  

***batch_size: int or 'auto', default='auto'***  

Size of minibatches for stochastic optimizers.
If 'auto', batch_size=min(200, n_samples).  

***learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'***

Learning rate schedule for weight updates.
'constant': Learning rate remains constant.
'invscaling': Gradually decreases the learning rate at each time step.
'adaptive': Keeps the learning rate constant as long as training loss decreases.  

***learning_rate_init: float, default=0.001***  

Initial learning rate used. Controls the step-size in updating weights.  

***power_t: float, default=0.5***  

Exponent for inverse scaling learning rate.  

***max_iter: int, default=200***  

Maximum number of iterations. The solver iterates until convergence or this number of iterations.  

***shuffle: bool, default=True***  

Whether to shuffle samples in each iteration.  

***random_state: int, RandomState instance, default=None***

Determines random number generation for weights and bias initialization. we need to initaillize the weights so that we can start training models.  

***tol: float, default=1e-4***

Tolerance for the optimization. When the loss does not improve by at least tol for n_iter_no_change consecutive iterations, convergence is considered to be reached.  

***verbose: bool, default=False***  

Whether to print progress messages to stdout.  

***warm_start: bool, default=False***  

When set to True, reuse the solution of the previous call to fit as initialization.

***early_stopping: bool, default=False***

Whether to use early stopping to terminate training when validation score is not improving.

***validation_fraction: float, default=0.1***

The proportion of training data to set aside as validation set for early stopping.


### Assignment 2
Finish the code below to train a neural network model for multi-class classification with the data you just processed. Depends on the computation power of your machine, you will need to adjust the number of hidden layers and neurons in each layer to achieve good performance. Also, the amount of training data you use depends on the computation power of your machine, use suitable size of training data. 

Use a sets of hyperparameter settings different from what's giving in the example code (keep verbose=True so that you can see the progress of the training process). You can try different learning rates, batch sizes, and activation functions. You can also try different architectures, such as adding more hidden layers or using convolutional neural networks.  

***Push your code to github and submit the link to your repository.***  

***Submit a document to canvas and explain why you set each parameter the way you choose, also describe how many training data you used.***

### Assignment 2: Explanation - Hyperparameter Settings and Training Data

**Submit a document to Canvas explaining why you set each parameter the way you chose, and describe how many training data samples you used.**

#### Training Data Used:
- **Total training samples**: [This will be printed when you run the code - check the output]
- **Number of features**: [Number of Sentinel-2 bands used]
- **Chip size**: 128×128 pixels
- **Label distribution**: [This will show the distribution across classes 0, 1, 2, 3]

#### Hyperparameter Justification:

**1. `hidden_layer_sizes=(128, 64)`**
- **Rationale**: Two hidden layers provide sufficient model capacity to learn complex patterns in multi-spectral remote sensing data. Starting with 128 neurons allows the model to capture rich feature representations, while the second layer with 64 neurons helps refine these features. This architecture balances model complexity with computational efficiency.

**2. `activation='relu'`**
- **Rationale**: ReLU (Rectified Linear Unit) is the standard choice for hidden layers as it helps mitigate the vanishing gradient problem, enables faster training, and has been shown to work well with remote sensing data. It introduces non-linearity needed for complex classification tasks.

**3. `solver='adam'`**
- **Rationale**: Adam optimizer is well-suited for this task as it adapts learning rates for each parameter, works efficiently with large datasets, and typically converges faster than SGD. It's particularly good for neural networks with many parameters.

**4. `alpha=0.001`**
- **Rationale**: Increased regularization (compared to default 0.0001) helps prevent overfitting, which is important when working with limited training data or when the model has sufficient capacity. This L2 penalty helps the model generalize better to unseen data.

**5. `batch_size=256`**
- **Rationale**: A moderate batch size provides a good balance between training stability and computational efficiency. Larger batches provide more stable gradient estimates, while smaller batches allow for more frequent weight updates. 256 is a good compromise for this dataset size.

**6. `learning_rate='adaptive'`**
- **Rationale**: Adaptive learning rate automatically adjusts if the training loss stops decreasing, which helps prevent the model from getting stuck in local minima and can improve convergence. This is particularly useful when the optimal learning rate is unknown.

**7. `learning_rate_init=0.01`**
- **Rationale**: A higher initial learning rate (compared to default 0.001) allows for faster initial learning. With adaptive learning rate scheduling, if this is too high, it will automatically adjust downward. This can speed up training without sacrificing final performance.

**8. `max_iter=500`**
- **Rationale**: Increased maximum iterations (from default 200) ensures the model has enough training epochs to converge, especially important with adaptive learning rates which may require more iterations to find optimal solutions.

**9. `early_stopping=True` and `validation_fraction=0.1`**
- **Rationale**: Early stopping prevents overfitting by monitoring validation performance and stopping training when validation loss stops improving. Using 10% of training data for validation provides a good estimate of generalization performance without significantly reducing training data.

**10. `random_state=42`**
- **Rationale**: Setting a random seed ensures reproducibility of results, allowing for consistent model training across different runs.


In [ ]:
import os
import numpy as np
from osgeo import gdal
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler  # For feature scaling
import joblib  # For saving the trained model

# =======================
# Data Preprocessing
# =======================

# Define chip size (adjust based on computation power: 64, 128, or 256)
chip_size = 128

# Directory containing training data
train_dir = './Lab6_data/train_Tulsa_Assignment1'

# Get all S2 and DW image pairs
s2_files = [f for f in os.listdir(train_dir) if f.endswith('_s2.tif')]
dw_files = [f for f in os.listdir(train_dir) if f.endswith('_dw.tif')]

# Sort to ensure matching pairs
s2_files.sort()
dw_files.sort()

print(f"Found {len(s2_files)} training image pairs")

# Lists to store all chips
all_X_chips = []
all_y_chips = []

# Loop through the image files
for s2_file, dw_file in zip(s2_files, dw_files):
    s2_path = os.path.join(train_dir, s2_file)
    dw_path = os.path.join(train_dir, dw_file)
    
    print(f"Processing: {s2_file} and {dw_file}")
    
    # Open S2 image (features)
    s2_ds = gdal.Open(s2_path)
    s2_array = s2_ds.ReadAsArray()  # Shape: (bands, height, width)
    s2_ds = None
    
    # Open DW label (ground truth)
    dw_ds = gdal.Open(dw_path)
    dw_array = dw_ds.ReadAsArray()  # Shape: (height, width) or (1, height, width)
    dw_ds = None
    
    # Handle single band label
    if len(dw_array.shape) == 3:
        dw_array = dw_array[0, :, :]
    
    # Get image dimensions
    n_bands, height, width = s2_array.shape
    
    # Chip the images
    for i in range(0, height, chip_size):
        for j in range(0, width, chip_size):
            # Extract chip boundaries
            i_end = min(i + chip_size, height)
            j_end = min(j + chip_size, width)
            
            # Extract chip from S2 image
            s2_chip = s2_array[:, i:i_end, j:j_end]
            # Reshape to (pixels, bands)
            s2_chip_flat = s2_chip.transpose(1, 2, 0).reshape(-1, n_bands)
            
            # Extract chip from DW label
            dw_chip = dw_array[i:i_end, j:j_end]
            # Flatten to 1D
            dw_chip_flat = dw_chip.flatten()
            
            # Only keep chips that are the full size (optional: can keep all chips)
            if s2_chip_flat.shape[0] == chip_size * chip_size:
                all_X_chips.append(s2_chip_flat)
                all_y_chips.append(dw_chip_flat)

# Concatenate all chips
X = np.concatenate(all_X_chips, axis=0)
y = np.concatenate(all_y_chips, axis=0)

print(f"Total training samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Label distribution: {np.bincount(y.astype(int))}")

# Feature scaling (important for neural networks)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initialize the neural network classifier with different hyperparameters
nn_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # Two hidden layers with 128 and 64 neurons
    activation='relu',
    solver='adam',
    alpha=0.001,  # Increased regularization
    batch_size=256,  # Explicit batch size
    learning_rate='adaptive',  # Adaptive learning rate
    learning_rate_init=0.01,  # Higher initial learning rate
    max_iter=500,  # More iterations
    random_state=42,
    tol=1e-4,
    verbose=True,  # Enable verbose output
    early_stopping=True,  # Enable early stopping
    validation_fraction=0.1  # 10% validation set
)

# Train the neural network model
print("Training Neural Network model...")
nn_model.fit(X_scaled, y)
print("Neural Network model trained.")

# Save the trained model and scaler
joblib.dump(nn_model, './neural_network_model.pkl')
joblib.dump(scaler, './scaler.pkl')

print("Neural Network model and scaler saved successfully.")


## Step 4: Running inference with the trained model and save the result
This step is similar to lab 6, be careful not to forget scaling the input data with the same scaling parameter. Save the inference result.
### Assignment 3
Finish the code below for running inference with the trained model and save the result.  

***Push your code to github***

In [ ]:
import os
import numpy as np
from osgeo import gdal
from sklearn.preprocessing import StandardScaler
import joblib

# =======================
# Paths and Model Loading
# =======================

# Path to the saved model and scaler
model_path = './neural_network_model.pkl'
scaler_path = './scaler.pkl'

# Load the trained neural network model and scaler using joblib
nn_model = joblib.load(model_path)
scaler = joblib.load(scaler_path)
print("Model and scaler loaded successfully.")

# =======================
# Input Data Preparation
# =======================

# Directory containing new images for inference
test_dir = './Lab6_data/Test_AL_Assignment4'

# Directory where prediction outputs will be saved
output_dir = './predictions'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Get all S2 test images
s2_test_files = [f for f in os.listdir(test_dir) if f.endswith('_s2.tif')]
s2_test_files.sort()

print(f"Found {len(s2_test_files)} test images")

# Process each test image
for s2_file in s2_test_files:
    s2_path = os.path.join(test_dir, s2_file)
    print(f"Processing: {s2_file}")
    
    # Open S2 image
    s2_ds = gdal.Open(s2_path)
    
    # Get georeferencing information
    geotransform = s2_ds.GetGeoTransform()
    projection = s2_ds.GetProjection()
    n_bands = s2_ds.RasterCount
    width = s2_ds.RasterXSize
    height = s2_ds.RasterYSize
    
    # Read all bands
    s2_array = s2_ds.ReadAsArray()  # Shape: (bands, height, width)
    s2_ds = None
    
    # Reshape to (pixels, bands) for prediction
    s2_reshaped = s2_array.transpose(1, 2, 0).reshape(-1, n_bands)
    
    # Normalize the input data using the same scaler used during training
    image_array_scaled = scaler.transform(s2_reshaped)
    
    # Make predictions
    print("Making predictions...")
    predictions = nn_model.predict(image_array_scaled)
    
    # Reshape predictions back to image dimensions
    predictions_2d = predictions.reshape(height, width).astype(np.uint8)
    
    # =======================
    # Save the Prediction as geo-referenced tif file
    # =======================
    
    # Create output filename
    output_filename = s2_file.replace('_s2.tif', '_prediction.tif')
    output_path = os.path.join(output_dir, output_filename)
    
    # Create output GeoTIFF
    driver = gdal.GetDriverByName('GTiff')
    out_ds = driver.Create(output_path, width, height, 1, gdal.GDT_Byte)
    out_ds.SetGeoTransform(geotransform)
    out_ds.SetProjection(projection)
    out_band = out_ds.GetRasterBand(1)
    out_band.WriteArray(predictions_2d)
    out_band.SetNoDataValue(0)
    out_band = None
    out_ds = None
    
    print(f"Prediction saved to: {output_path}")

print("All predictions completed!")


### Assignment 4: Discussion - Macro vs. Micro Averaging and DW Labels

**Submit a Word document discussing:**
1. The pros and cons of macro-averaging and micro-averaging for multi-class classification accuracy assessment
2. Which one makes more sense in this region and why
3. The pros and cons of using DW labels for multi-class classification accuracy assessment

---

#### 1. Macro-Averaging vs. Micro-Averaging: Pros and Cons

**Macro-Averaging:**

**Pros:**
- Treats all classes equally, regardless of class frequency
- Provides insight into model performance for minority classes
- Better for imbalanced datasets where you care about all classes equally
- More interpretable when class distribution is highly skewed
- Useful when rare classes are important (e.g., rare land cover types)

**Cons:**
- Can be misleading if classes are highly imbalanced (a rare class with poor performance will have equal weight)
- May not reflect overall model performance if one class dominates the dataset
- Sensitive to performance on small classes
- May not represent the actual user experience if most pixels belong to common classes

**Micro-Averaging:**

**Pros:**
- Reflects overall model performance across all samples
- More representative of actual classification accuracy when classes are imbalanced
- Better represents the user experience (most pixels will be from common classes)
- Less sensitive to performance on rare classes
- Equivalent to overall accuracy in multi-class classification

**Cons:**
- Dominated by majority classes in imbalanced datasets
- Poor performance on rare classes may be masked
- May not reveal important classification errors for minority classes
- Less informative when you need to ensure all classes are classified well

---

#### 2. Which Makes More Sense for This Region?

**Analysis should consider:**
- **Class distribution**: Check the support values in the accuracy assessment results to see if classes are balanced or imbalanced
- **Application context**: 
  - If all land cover types are equally important (e.g., environmental monitoring), macro-averaging may be preferred
  - If the goal is overall classification accuracy and common classes dominate, micro-averaging may be more appropriate
- **Regional characteristics**: 
  - Urban areas: Built-up and other classes may dominate → micro-averaging may be more representative
  - Rural/forested areas: Trees may dominate → consider both metrics
  - Mixed landscapes: Macro-averaging ensures all classes are evaluated fairly

**Recommendation**: [Based on your results, discuss which metric better represents the model's performance for your specific region and application]

---

#### 3. Pros and Cons of Using DW Labels for Accuracy Assessment

**Pros:**
- **Abundance**: DW provides extensive labeled data covering large geographic areas and temporal periods
- **Consistency**: Labels are generated using a consistent algorithm, reducing human labeling inconsistencies
- **Temporal coverage**: Near real-time availability allows for assessment across different time periods
- **Cost-effective**: No need for expensive manual labeling or field surveys
- **Spatial coverage**: Global coverage enables assessment in diverse geographic regions
- **Standardized format**: Consistent data structure and classification scheme

**Cons:**
- **Algorithm-generated labels**: DW labels are themselves predictions from a deep learning model, not ground truth
- **Accuracy limitations**: DW labels have inherent errors and uncertainties (typically 60-80% accuracy depending on class)
- **Circular validation**: Using DW labels to assess a model trained on DW labels creates a circular reference
- **Class confusion**: Some classes may be systematically misclassified in DW (e.g., confusion between similar classes)
- **Temporal mismatch**: DW labels may not perfectly align with the Sentinel-2 image acquisition time
- **Not suitable for absolute accuracy**: Cannot determine true model performance, only relative performance compared to DW
- **Bias propagation**: Any systematic errors in DW will be reflected in the assessment

**Best Practices:**
- Use DW labels for relative model comparison and development
- Validate critical results with manually labeled reference data when possible
- Acknowledge limitations when reporting results
- Consider DW labels as "weak labels" rather than ground truth
- Use for model development and hyperparameter tuning, but validate with independent data for final assessment

---

**Note**: After running the accuracy assessment code, review the actual results (class distribution, per-class metrics) to provide specific examples and quantitative support for your discussion points.


## Step 5: Multi-class classification accuracy assessment
When doing accuracy assessment for multi-class classification, the general idea is to calculate the accuracy of each class separately and then take the average of those accuracies. Calculating the accuracy of each class separately means that we are doingf binary classification when calculating the accuracy of each class. For example, when calculating class1, only positive samples from class1 are considered as 1, all other pixels are 0. Similarly, when calculating class2, only positive samples from class2 are considered as 1, all other pixels are 0. We repeat this process for all classes and then take the average of the accuracies to get the overall accuracy.

However, when taking average, we need to consider if using macro-averaging or micro-averaging. Macro-averaging means that we calculate the accuracy of each class and then take the average of those accuracies. Micro-averaging means that we calculate the overall accuracy by considering all samples together, regardless of their class.  
More specifically, Macro averaging calculates metrics independently for each class and then takes the average (unweighted mean) across classes. For any metric, the macro-average is calculated as: 

$$\text{Macro-average} = \frac{1}{K}\sum_{k=1}^{K} \text{Accuracy}_k$$

Micro averaging calculates the overall accuracy by considering all samples together, regardless of their class. For any metric, micro average will sum up the true positives (TP), false positives (FP), and false negatives (FN) across all classes. And then compute the metric using these aggregated sums.

Below is example code of how to use scikit-learn to calculate accuracy for multi-class classification.

### Assignment 4: Multi-class classification accuracy assessment
Refer to the code below for multi-class classification accuracy assessment using scikit-learn. Chip the data I provided and Perform accuracy assessment using both macro-averaging and micro-averaging. Save the result to txt file for each chip. So you should have two txt files, one for macro-averaging and one for micro-averaging. you don't need to print the result to the console. I'm doing it for demonstration purposes.

***push your code to your GitHub repository***  

***In a word document, discuss the pros and cons of macro-averaging and micro-averaging for multi-class classification accuracy assessment for the data I provided. Which one makes more sense in this region and why?***  

***Discuss the pro and cons of using DW labels for multi-class classification accuracy assessment.***

In [ ]:
import numpy as np
import rasterio
from osgeo import gdal
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.metrics import jaccard_score
import os

# =======================
# Paths and Data Loading
# =======================

# Directory containing test data
test_dir = './Lab6_data/Test_AL_Assignment4'

# Directory containing predictions
pred_dir = './predictions'

# Get all label and prediction files
label_files = [f for f in os.listdir(test_dir) if f.endswith('_dw.tif')]
label_files.sort()

# Output directory for accuracy assessment results
output_dir = './accuracy_assessment'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# List of classes (ensure these are all the possible classes in your data)
classes = [0, 1, 2, 3]  # 0=other, 1=water, 2=trees, 3=built-up

# Process each chip/image pair
for label_file in label_files:
    # Construct paths
    label_path = os.path.join(test_dir, label_file)
    
    # Find corresponding prediction file
    # Try multiple naming patterns
    base_name = label_file.replace('_dw.tif', '').replace('_DW_small', '').replace('DW_small', '')
    
    # Try different naming patterns
    possible_pred_names = [
        label_file.replace('_dw.tif', '_s2_prediction.tif'),
        label_file.replace('_dw.tif', '_prediction.tif'),
        base_name + '_s2_prediction.tif',
        base_name + '_prediction.tif',
    ]
    
    pred_path = None
    for pred_name in possible_pred_names:
        test_path = os.path.join(pred_dir, pred_name)
        if os.path.exists(test_path):
            pred_path = test_path
            break
    
    # If still not found, try to find any prediction file (for single file case)
    if pred_path is None or not os.path.exists(pred_path):
        pred_files = [f for f in os.listdir(pred_dir) if f.endswith('_prediction.tif')]
        if pred_files:
            pred_path = os.path.join(pred_dir, pred_files[0])
    
    if not os.path.exists(pred_path):
        print(f"Warning: Prediction file not found for {label_file}, skipping...")
        continue
    
    print(f"Processing: {label_file}")
    
    # Read in the label and prediction rasters
    label_ds = gdal.Open(label_path)
    label_array = label_ds.ReadAsArray()
    label_ds = None
    
    pred_ds = gdal.Open(pred_path)
    pred_array = pred_ds.ReadAsArray()
    pred_ds = None
    
    # Handle single band arrays
    if len(label_array.shape) == 3:
        label_array = label_array[0, :, :]
    if len(pred_array.shape) == 3:
        pred_array = pred_array[0, :, :]
    
    # Flatten the arrays to 1D arrays
    label_flat = label_array.flatten().astype(int)
    pred_flat = pred_array.flatten().astype(int)
    
    # Remove any no-data values (if applicable)
    valid_mask = (label_flat >= 0) & (label_flat <= 3) & (pred_flat >= 0) & (pred_flat <= 3)
    label_flat = label_flat[valid_mask]
    pred_flat = pred_flat[valid_mask]
    
    # =======================
    # Calculate Metrics
    # =======================
    
    # Calculate Overall Accuracy
    oa = accuracy_score(label_flat, pred_flat)
    
    # Calculate Precision, Recall, F1 Score per class
    precision, recall, f1_score, support = precision_recall_fscore_support(
        label_flat, pred_flat, labels=classes, average=None, zero_division=0
    )
    
    # Calculate Macro Averaged Metrics
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        label_flat, pred_flat, average='macro', zero_division=0
    )
    
    # Calculate Micro Averaged Metrics
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        label_flat, pred_flat, average='micro', zero_division=0
    )
    
    # Calculate Jaccard Score (IoU) per class
    jaccard_per_class = jaccard_score(label_flat, pred_flat, labels=classes, average=None, zero_division=0)
    jaccard_macro = jaccard_score(label_flat, pred_flat, labels=classes, average='macro', zero_division=0)
    jaccard_micro = jaccard_score(label_flat, pred_flat, labels=classes, average='micro', zero_division=0)
    
    # =======================
    # Save Macro-Averaged Results
    # =======================
    
    base_name = os.path.splitext(label_file)[0]
    macro_output_path = os.path.join(output_dir, f'{base_name}_macro_accuracy.txt')
    
    with open(macro_output_path, 'w') as f:
        f.write("=" * 60 + "\n")
        f.write("MACRO-AVERAGED ACCURACY ASSESSMENT\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Image: {label_file}\n\n")
        
        f.write("Overall Accuracy: {:.4f}\n\n".format(oa))
        
        f.write("Per-Class Metrics:\n")
        f.write("-" * 60 + "\n")
        for idx, cls in enumerate(classes):
            class_names = {0: 'Other', 1: 'Water', 2: 'Trees', 3: 'Built-up'}
            f.write(f"\nClass {cls} ({class_names.get(cls, 'Unknown')}):\n")
            f.write(f"  Precision: {precision[idx]:.4f}\n")
            f.write(f"  Recall:    {recall[idx]:.4f}\n")
            f.write(f"  F1 Score:  {f1_score[idx]:.4f}\n")
            f.write(f"  Jaccard:   {jaccard_per_class[idx]:.4f}\n")
            f.write(f"  Support:   {support[idx]}\n")
        
        f.write("\n" + "-" * 60 + "\n")
        f.write("Macro-Averaged Metrics:\n")
        f.write(f"  Precision: {precision_macro:.4f}\n")
        f.write(f"  Recall:    {recall_macro:.4f}\n")
        f.write(f"  F1 Score:  {f1_macro:.4f}\n")
        f.write(f"  Jaccard:   {jaccard_macro:.4f}\n")
    
    # =======================
    # Save Micro-Averaged Results
    # =======================
    
    micro_output_path = os.path.join(output_dir, f'{base_name}_micro_accuracy.txt')
    
    with open(micro_output_path, 'w') as f:
        f.write("=" * 60 + "\n")
        f.write("MICRO-AVERAGED ACCURACY ASSESSMENT\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Image: {label_file}\n\n")
        
        f.write("Overall Accuracy: {:.4f}\n\n".format(oa))
        
        f.write("Per-Class Metrics:\n")
        f.write("-" * 60 + "\n")
        for idx, cls in enumerate(classes):
            class_names = {0: 'Other', 1: 'Water', 2: 'Trees', 3: 'Built-up'}
            f.write(f"\nClass {cls} ({class_names.get(cls, 'Unknown')}):\n")
            f.write(f"  Precision: {precision[idx]:.4f}\n")
            f.write(f"  Recall:    {recall[idx]:.4f}\n")
            f.write(f"  F1 Score:  {f1_score[idx]:.4f}\n")
            f.write(f"  Jaccard:   {jaccard_per_class[idx]:.4f}\n")
            f.write(f"  Support:   {support[idx]}\n")
        
        f.write("\n" + "-" * 60 + "\n")
        f.write("Micro-Averaged Metrics:\n")
        f.write(f"  Precision: {precision_micro:.4f}\n")
        f.write(f"  Recall:    {recall_micro:.4f}\n")
        f.write(f"  F1 Score:  {f1_micro:.4f}\n")
        f.write(f"  Jaccard:   {jaccard_micro:.4f}\n")
    
    print(f"Accuracy assessment saved: {macro_output_path}")
    print(f"Accuracy assessment saved: {micro_output_path}")

print("All accuracy assessments completed!")

